**Installation**

In [ ]:
%pip install unsloth

**Load the model**

In [2]:

from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it",
    max_seq_length = 1024,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

**Attach LoRA adapters**

In [3]:
model = FastModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

**Load the dataset**

In [4]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca", split="train")

#format the dataset into Gemma's chat template.

In [5]:
def format_prompt(example):
    if example["input"]:
        user_text = example["instruction"] + "\n\n" + example["input"]
    else:
        user_text = example["instruction"]

    convo = [
        {"role": "user", "content": user_text},
        {"role": "model", "content": example["output"]},
    ]

    text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

**Set up the trainer**

In [6]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 1024,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
        fp16 = False,

    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/52002 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


**Start training**

In [7]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 52,002 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 13,045,760 of 1,012,931,712 (1.29% trained)


Step,Training Loss
1,6.901678
2,5.753686
3,6.042951
4,6.153594
5,4.329770
6,3.564218
7,3.169136
8,2.724851
9,2.389138
10,2.524501


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


**Test the fine-tuned model**

In [9]:
from transformers import TextStreamer

FastModel.for_inference(model)

messages = [
    {"role": "user", "content": "What is the capital of France??"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids=inputs, streamer=streamer, max_new_tokens=128)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The capital of France is Paris.<end_of_turn>


In [10]:
model.save_pretrained("gemma3-alpaca-lora")
tokenizer.save_pretrained("gemma3-alpaca-lora")

Unsloth: Restored added_tokens_decoder metadata in gemma3-alpaca-lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma3-alpaca-lora.


('gemma3-alpaca-lora/tokenizer_config.json',
 'gemma3-alpaca-lora/chat_template.jinja',
 'gemma3-alpaca-lora/tokenizer.json')

In [13]:
from huggingface_hub import notebook_login
notebook_login()

In [14]:
model.push_to_hub("wazym/gemma3-alpaca-lora", token=True)
tokenizer.push_to_hub("wazym/gemma3-alpaca-lora", token=True)

README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|1         |  559kB / 52.2MB            

Saved model to https://huggingface.co/wazym/gemma3-alpaca-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpyuv0bm5k/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /tmp/tmpyuv0bm5k.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pyuv0bm5k/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...mpyuv0bm5k/tokenizer.json:  96%|#########5| 31.9MB / 33.4MB            